# eval.ipynb

**Langkah 4: evaluasi pada set tes yang disisihkan.**

Bergantung pada `data.ipynb` dan `metrics.ipynb` (dimuat dengan `%run`). Model dimuat dengan `compile=False` (head output adalah layer standar, jadi tidak perlu scope custom-object). Melaporkan Accuracy, F1, Jaccard (IoU), Recall, Precision, dan Boundary IoU per gambar, menulis strip perbandingan ke `results/`, dan memvisualisasikan kasus terbaik dan terburuk.

In [ ]:
# Muat modul yang dibutuhkan notebook ini.
# _DEMO=False menekan sel demo/self-check mereka selama %run.
_DEMO = False
%run data.ipynb
%run metrics.ipynb
_DEMO = True

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import cv2
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, f1_score, jaccard_score, precision_score, recall_score,
)

METRIC_NAMES = ["Accuracy", "F1", "Jaccard", "Recall", "Precision", "BoundaryIoU"]

### Strip hasil

Original | Ground Truth | Prediksi | Masked. Mask dan prediksi di-binarisasi ke 0/1 sebelum diskalakan ke 0/255, yang memperbaiki overflow lama yang menggelapkan panel ground-truth.

In [ ]:
def save_results(image, mask, y_pred, save_path):
    """
    Simpan strip perbandingan: Original | Ground Truth | Prediksi | Masked.

    image  : HxWx3 uint8 BGR
    mask   : HxW   uint8 (ground truth biner 0/1 atau 0/255)
    y_pred : HxW   prediksi biner (0/1)
    """
    h = image.shape[0]
    line = np.ones((h, 10, 3), dtype=np.uint8) * 128

    mask_bin = (mask > 0).astype(np.uint8)         # 0/1, robust terhadap mask 0/1 atau 0/255
    pred_bin = (y_pred > 0).astype(np.uint8)       # 0/1

    mask_vis = np.stack([mask_bin] * 3, axis=-1) * 255
    pred_vis = np.stack([pred_bin] * 3, axis=-1) * 255
    masked   = (image * np.stack([pred_bin] * 3, axis=-1)).astype(np.uint8)

    strip = np.concatenate([image, line, mask_vis, line, pred_vis, line, masked], axis=1)
    cv2.imwrite(save_path, strip)

### Visualisasi

Bar chart metrik rata-rata, box plot per gambar, sampel prediksi, dan panel terbaik-vs-terburuk berdasarkan peringkat Jaccard.

In [ ]:
def _predict_mask(model, image_bgr):
    """Standardisasi -> normalisasi -> prediksi -> threshold. Mengembalikan 0/1 uint8 HxW."""
    img = standardize_image(image_bgr)
    x = np.expand_dims(img.astype(np.float32) / 255.0, axis=0)
    pred = model.predict(x, verbose=0)[0]
    pred = np.squeeze(pred, axis=-1)
    return (pred > 0.5).astype(np.uint8)

In [ ]:
def visualize_metrics(score_mean, save_dir="plots"):
    """Bar chart horizontal metrik evaluasi rata-rata. plots/06_evaluation_metrics.png."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    create_dir(save_dir)
    names  = ["Accuracy", "Skor F1", "Jaccard (IoU)", "Recall", "Precision", "Boundary IoU"]
    values = list(score_mean)
    colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336", "#00897B"]

    fig, ax = plt.subplots(figsize=(9, 5.5))
    bars = ax.barh(names, values, color=colors, edgecolor="white", linewidth=1.2, height=0.6)
    for bar, val in zip(bars, values):
        ax.text(min(val + 0.01, 0.99), bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", ha="left", fontsize=11, fontweight="bold")
    ax.set_xlim(0, 1.08)
    ax.set_xlabel("Skor", fontsize=12)
    ax.set_title("Metrik Evaluasi - Set Tes", fontsize=14, fontweight="bold")
    ax.axvline(0.9, color="gray", linestyle="--", linewidth=1, alpha=0.6, label="Referensi 0.90")
    ax.legend(fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="x", alpha=0.3)
    ax.invert_yaxis()

    plt.tight_layout()
    out = os.path.join(save_dir, "06_evaluation_metrics.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Metrik evaluasi disimpan: {out}")

In [ ]:
def visualize_predictions(sample_paths, model, save_dir="plots", n_samples=6, fname="07_prediction_samples.png", title="Sampel Prediksi - DeepLabV3+"):
    """Grid baris: Original | Ground Truth | Prediksi | Hasil Masked."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    create_dir(save_dir)
    n_samples = min(n_samples, len(sample_paths))
    if n_samples == 0:
        return

    fig, axes = plt.subplots(n_samples, 4, figsize=(16, n_samples * 3.2))
    if n_samples == 1:
        axes = axes[np.newaxis, :]

    for col, t in enumerate(["Original", "Ground Truth", "Prediksi", "Hasil Masked"]):
        axes[0, col].set_title(t, fontsize=12, fontweight="bold", pad=8)

    for row, (x_path, y_path) in enumerate(sample_paths[:n_samples]):
        img  = standardize_image(cv2.imread(x_path, cv2.IMREAD_COLOR))
        mask = standardize_mask(cv2.imread(y_path, cv2.IMREAD_GRAYSCALE))
        pred = _predict_mask(model, img)
        masked = img * np.stack([pred, pred, pred], axis=-1)

        axes[row, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[row, 1].imshow(mask, cmap="gray")
        axes[row, 2].imshow(pred, cmap="gray")
        axes[row, 3].imshow(cv2.cvtColor(masked.astype(np.uint8), cv2.COLOR_BGR2RGB))

        name = os.path.basename(x_path).split(".")[0]
        axes[row, 0].set_ylabel(name, fontsize=8, rotation=0, labelpad=60, va="center")
        for col in range(4):
            axes[row, col].axis("off")

    fig.suptitle(title, fontsize=15, fontweight="bold")
    plt.tight_layout()
    out = os.path.join(save_dir, fname)
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] {title} disimpan: {out}")

In [ ]:
def visualize_metrics_distribution(score_df, save_dir="plots"):
    """Box plot metrik per gambar. plots/08_metrics_distribution.png."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    create_dir(save_dir)
    cols   = ["Accuracy", "F1", "Jaccard", "Recall", "Precision", "BoundaryIoU"]
    data   = [score_df[c].values for c in cols]
    colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336", "#00897B"]

    fig, ax = plt.subplots(figsize=(11, 6))
    bp = ax.boxplot(data, patch_artist=True, medianprops={"color": "white", "linewidth": 2})
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax.set_xticklabels(cols, fontsize=10)
    ax.set_ylabel("Skor", fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.set_title("Distribusi Metrik Per Gambar - Set Tes", fontsize=14, fontweight="bold")
    ax.axhline(0.9, color="gray", linestyle="--", linewidth=1, alpha=0.6)
    ax.grid(axis="y", alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    out = os.path.join(save_dir, "08_metrics_distribution.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Distribusi metrik disimpan: {out}")

In [ ]:
def visualize_best_worst(score_df, pairs, model, save_dir="plots", k=3):
    """
    Visualisasikan k prediksi terbaik dan k terburuk berdasarkan skor Jaccard,
    ditumpuk dalam satu figur. plots/08b_best_worst.png. Membantu menemukan
    mode kegagalan sistematis.
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    create_dir(save_dir)
    if len(score_df) == 0:
        return

    path_by_name = {os.path.splitext(os.path.basename(x))[0]: (x, y) for x, y in pairs}
    ordered = score_df.sort_values("Jaccard", ascending=False)
    k = min(k, len(ordered) // 2 if len(ordered) >= 2 else 1)
    best = ordered.head(k)
    worst = ordered.tail(k)
    rows = list(best.itertuples()) + list(worst.itertuples())
    labels = [f"TERBAIK  J={r.Jaccard:.3f}" for r in best.itertuples()] + \
             [f"TERBURUK J={r.Jaccard:.3f}" for r in worst.itertuples()]

    n = len(rows)
    fig, axes = plt.subplots(n, 4, figsize=(16, n * 3.0))
    if n == 1:
        axes = axes[np.newaxis, :]
    for col, t in enumerate(["Original", "Ground Truth", "Prediksi", "Masked"]):
        axes[0, col].set_title(t, fontsize=12, fontweight="bold", pad=8)

    for row, (r, lbl) in enumerate(zip(rows, labels)):
        name = r.Image
        if name not in path_by_name:
            continue
        x_path, y_path = path_by_name[name]
        img  = standardize_image(cv2.imread(x_path, cv2.IMREAD_COLOR))
        mask = standardize_mask(cv2.imread(y_path, cv2.IMREAD_GRAYSCALE))
        pred = _predict_mask(model, img)
        masked = img * np.stack([pred, pred, pred], axis=-1)

        axes[row, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[row, 1].imshow(mask, cmap="gray")
        axes[row, 2].imshow(pred, cmap="gray")
        axes[row, 3].imshow(cv2.cvtColor(masked.astype(np.uint8), cv2.COLOR_BGR2RGB))
        color = "#2E7D32" if lbl.startswith("TERBAIK") else "#C62828"
        axes[row, 0].set_ylabel(lbl, fontsize=9, rotation=0, labelpad=70,
                                va="center", color=color, fontweight="bold")
        for col in range(4):
            axes[row, col].axis("off")

    fig.suptitle(f"Prediksi Terbaik vs Terburuk ({k} teratas / {k} terbawah berdasarkan Jaccard)",
                 fontsize=15, fontweight="bold")
    plt.tight_layout()
    out = os.path.join(save_dir, "08b_best_worst.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Prediksi terbaik/terburuk disimpan: {out}")

### Loop evaluasi

Menjalankan inferensi pada set tes, menulis strip, dan mengembalikan DataFrame skor per gambar.

In [ ]:
def evaluate(model, test_x, test_y, results_dir="results"):
    """Jalankan evaluasi lengkap, tulis strip hasil, kembalikan DataFrame skor."""
    create_dir(results_dir)
    rows = []
    for x_path, y_path in tqdm(zip(test_x, test_y), total=len(test_x)):
        name  = os.path.splitext(os.path.basename(x_path))[0]
        image = standardize_image(cv2.imread(x_path, cv2.IMREAD_COLOR))
        mask  = standardize_mask(cv2.imread(y_path, cv2.IMREAD_GRAYSCALE))

        y_pred = _predict_mask(model, image)

        save_results(image, mask, y_pred, os.path.join(results_dir, f"{name}.png"))

        mask_flat = (mask.flatten() > 0).astype(np.int32)
        pred_flat = y_pred.flatten().astype(np.int32)

        rows.append([
            name,
            accuracy_score(mask_flat, pred_flat),
            f1_score(mask_flat, pred_flat, labels=[0, 1], average="binary", zero_division=0),
            jaccard_score(mask_flat, pred_flat, labels=[0, 1], average="binary", zero_division=1),
            recall_score(mask_flat, pred_flat, labels=[0, 1], average="binary", zero_division=1),
            precision_score(mask_flat, pred_flat, labels=[0, 1], average="binary", zero_division=1),
            boundary_iou(mask, y_pred),
        ])

    return pd.DataFrame(rows, columns=["Image"] + METRIC_NAMES)

### Jalankan evaluasi

Muat checkpoint, evaluasi, cetak metrik rata-rata, dan tampilkan semua figur evaluasi.

In [ ]:
from IPython.display import Image, display

create_dir("results"); create_dir("plots")
MODEL_PATH = os.path.join("files", "model.h5")
if not os.path.exists(MODEL_PATH):
    raise SystemExit("Tidak ada checkpoint. Jalankan train.ipynb terlebih dahulu.")
eval_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
test_x, test_y = load_processed_data(os.path.join("new_data", "test"))
print(f"Pasangan tes: {len(test_x)}")

score_df = evaluate(eval_model, test_x, test_y, results_dir="results")
score_mean = score_df[METRIC_NAMES].mean()
print("\nMetrik rata-rata:")
for name in METRIC_NAMES:
    print(f"  {name:<12}: {score_mean[name]:.4f}")

In [ ]:
visualize_metrics(score_mean.values, save_dir="plots")
display(Image("plots/06_evaluation_metrics.png"))
visualize_metrics_distribution(score_df, save_dir="plots")
display(Image("plots/08_metrics_distribution.png"))
pairs = list(zip(test_x, test_y))
visualize_predictions(pairs[:6], eval_model, save_dir="plots", n_samples=6)
display(Image("plots/07_prediction_samples.png"))
visualize_best_worst(score_df, pairs, eval_model, save_dir="plots", k=3)
display(Image("plots/08b_best_worst.png"))